In [1]:
# Parameters
base_path = "C:\\TCC_USP"
run_id = "20251126_163929"


In [2]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "06_preprocessing_real"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


BASE_PATH: C:\TCC_USP
PROC_PATH: C:\TCC_USP\data_processed


In [3]:
# Legacy Colab mount removido; paths definidos no setup.


In [4]:
# 3. Carregar dataset de notícias reais
import pandas as pd
news_file = os.path.join(RAW_PATH, "noticias_real.csv")
assert os.path.exists(news_file), f"Arquivo não encontrado: {news_file}"

df_news = pd.read_csv(news_file)
print("Shape inicial:", df_news.shape)
display(df_news.head())

Shape inicial: (632, 4)


,data,fonte,titulo,texto
0,2025-09-27,IGN,"""Apenas 1 em cada 10 mil jogadores encontraria...",Esqueceu ou decidiu não comprar a Pena? O game...
1,2025-09-27,Sapo.pt,Google Fotos vai ao Tinder copiar uma nova fun...,O Google Fotos poderá receber uma funcionalida...
2,2025-09-27,Observador.pt,Fogo em Oleiros dominado ao início da tarde,Fogo deflagrou por volta das 11:50 deste sábad...
3,2025-09-27,Abril.com.br,"Os maiores pecados de quem bebe vinho, segundo...",Vale conferir as dicas preparadas especialment...
4,2025-09-27,Metropoles.com,Alice Wegmann reage após gafe em cena de Vale ...,Cena de Vale Tudo foi detonada nas redes socia...


In [5]:
# 4. Padronizar colunas
df_news = df_news.rename(columns={
    "data": "data",
    "fonte": "fonte",
    "titulo": "titulo",
    "texto": "texto"
})

# Garantir formato de data
df_news["data"] = pd.to_datetime(df_news["data"], errors="coerce")

In [6]:
# 5. Pré-processamento de texto
import re, nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

stop_pt = set(stopwords.words("portuguese"))

def preprocess_text(t):
    if pd.isna(t):
        return ""
    t = re.sub(r"[^A-Za-z\u00C0-\u00FF\s]", " ", str(t)).lower()
    toks = [w for w in t.split() if w and w not in stop_pt]
    return " ".join(toks)

# Criar coluna limpa a partir de título + texto
df_news["texto_completo"] = (df_news["titulo"].fillna("") + " " + df_news["texto"].fillna("")).str.strip()
df_news["clean_text"] = df_news["texto_completo"].apply(preprocess_text)

print("✅ Pré-processamento concluído!")
display(df_news[["data","fonte","titulo","clean_text"]].head())

✅ Pré-processamento concluído!


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ander\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,data,fonte,titulo,clean_text
0,2025-09-27,IGN,"""Apenas 1 em cada 10 mil jogadores encontraria...",apenas cada mil jogadores encontraria jogou si...
1,2025-09-27,Sapo.pt,Google Fotos vai ao Tinder copiar uma nova fun...,google fotos vai tinder copiar nova funcionali...
2,2025-09-27,Observador.pt,Fogo em Oleiros dominado ao início da tarde,fogo oleiros dominado início tarde fogo deflag...
3,2025-09-27,Abril.com.br,"Os maiores pecados de quem bebe vinho, segundo...",maiores pecados bebe vinho segundo papa francê...
4,2025-09-27,Metropoles.com,Alice Wegmann reage após gafe em cena de Vale ...,alice wegmann reage após gafe cena vale tudo d...


In [7]:
# 6. Salvar dataset processado
news_out = os.path.join(PROC_PATH, "noticias_real_clean.csv")
df_news.to_csv(news_out, index=False, encoding="utf-8-sig")

print("Arquivo salvo em:", news_out)

Arquivo salvo em: C:\TCC_USP\data_processed\noticias_real_clean.csv


In [8]:
# Dependência já deve estar instalada via requirements.txt

In [9]:
# 8. Score de CREDIBILIDADE por fonte (regra simples e extensível)
cred_map = {
    # Alta
    "CVM": 1.00, "Comunicado ao Mercado": 0.95, "Site RI": 0.95,
    "Valor Econômico": 0.90, "Reuters": 0.90, "Reuters Brasil": 0.90, "Estadão": 0.88,
    "InfoMoney": 0.85, "Exame": 0.83,
    # Média
    "Portal do Investidor": 0.80, "UOL": 0.75, "G1": 0.75,
    # Desconhecidas/Outras
    "desconhecida": 0.60
}
df_news["source_credibility"] = df_news["fonte"].map(cred_map).fillna(0.70)

# Salvar versão estendida
news_out2 = os.path.join(PROC_PATH, "noticias_real_features.csv")
df_news.to_csv(news_out2, index=False, encoding="utf-8-sig")
print("🔁 Salvo com novelty & credibility:", news_out2)

🔁 Salvo com novelty & credibility: C:\TCC_USP\data_processed\noticias_real_features.csv
